# Python Tutorial: Clean Sample Metadata

**End goal:** load a sample sheet, check study balance, and make a simple plot.

This is the kind of small task Python is excellent at: read a table, clean it, summarize it, and save an output.

## Setup: load the tutorial data

Run this cell first. It uses the local files when they already exist and downloads the tiny tutorial data when you open the notebook in Colab or another fresh environment.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)
raw_base = 'https://raw.githubusercontent.com/Caffeinated-Code/Bioinformatics-Field-Guide/main/content/resources/week-02/data'
for file_name in ['samples.tsv', 'mini_transcripts.fasta', 'gene_counts.tsv', 'gene_annotations.tsv']:
    target = data_dir / file_name
    if not target.exists():
        urlretrieve(f"{raw_base}/{file_name}", target)

sorted(path.name for path in data_dir.iterdir())


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path('../data')
samples = pd.read_csv(data_dir / 'samples.tsv', sep='\t')
samples

## 1. Check the shape and columns

Before analysis, learn what you loaded.

In [ ]:
print(samples.shape)
print(samples.columns.tolist())
samples.dtypes

## 2. Clean simple text columns

Real metadata often has mixed capitalization or stray spaces. Practice the habit even on tiny data.

In [ ]:
clean = samples.copy()
for column in ['condition', 'batch', 'tissue']:
    clean[column] = clean[column].str.strip().str.lower()

clean

## 3. Summarize the study design

Ask two questions before modeling: are conditions balanced, and are batches mixed across conditions?

In [ ]:
condition_counts = clean['condition'].value_counts().sort_index()
batch_table = pd.crosstab(clean['batch'], clean['condition'])

print(condition_counts)
batch_table

## 4. Make a quick QC plot

A plot can reveal whether read depth is very different between groups.

In [ ]:
ax = clean.boxplot(column='reads_million', by='condition')
ax.set_title('Read depth by condition')
ax.set_xlabel('Condition')
ax.set_ylabel('Reads, millions')
plt.suptitle('')
plt.show()

## 5. Save a cleaned table

A useful script leaves behind a clean artifact.

In [ ]:
out = Path('outputs')
out.mkdir(exist_ok=True)
clean.to_csv(out / 'clean_samples.tsv', sep='\t', index=False)
out / 'clean_samples.tsv'

## Your turn

- Add a new column called `reads_group` with values `low` and `high`.
- Plot read depth by `batch` instead of `condition`.
- Filter the table to treated samples only.

**Confidence check:** you should be able to explain what each line changed in the metadata table.